In [0]:
import sys, os, subprocess

DBT_LIB = "/tmp/dbtlib"
if not os.path.exists(f"{DBT_LIB}/dbt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--ignore-installed",
                    "--target", DBT_LIB, "dbt-databricks>=1.8"], check=True)

env_check = dict(os.environ, PYTHONPATH=DBT_LIB, PYTHONNOUSERSITE="1")
r = subprocess.run([sys.executable, "-c", "from dbt.cli.main import cli; cli()", "--version"],
                   capture_output=True, text=True, env=env_check)
print(r.stdout or r.stderr[-2000:])

In [0]:
import os

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"
for d in ["models/staging", "models/intermediate", "models/marts", "macros", "snapshots", "seeds", "tests"]:
    os.makedirs(f"{DBT}/{d}", exist_ok=True)

files = {
"dbt_project.yml": """name: oil_shock
version: "1.0.0"
config-version: 2
profile: oil_shock

model-paths: ["models"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]
seed-paths: ["seeds"]
test-paths: ["tests"]

models:
  oil_shock:
    staging:
      +schema: silver
      +materialized: view
    intermediate:
      +schema: silver
      +materialized: view
    marts:
      +schema: gold
      +materialized: table
""",
"macros/generate_schema_name.sql": """{% macro generate_schema_name(custom_schema_name, node) -%}
  {%- if custom_schema_name is none -%}{{ target.schema }}{%- else -%}{{ custom_schema_name | trim }}{%- endif -%}
{%- endmacro %}
""",
".gitignore": "target/\nlogs/\ndbt_packages/\n",
}

for path, content in files.items():
    with open(f"{DBT}/{path}", "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
import os, sys, subprocess

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"
DBT_LIB = "/tmp/dbtlib"
PROFILES = "/tmp/dbt_profiles"
os.makedirs(PROFILES, exist_ok=True)

with open(f"{PROFILES}/profiles.yml", "w") as f:
    f.write("""oil_shock:
  target: prod
  outputs:
    prod:
      type: databricks
      catalog: oil_shock
      schema: silver
      host: dbc-050dc881-f83e.cloud.databricks.com
      http_path: /sql/1.0/warehouses/42944d45ee999e1c
      token: "{{ env_var('DBT_DATABRICKS_TOKEN') }}"
      threads: 4
""")

env = dict(os.environ,
           PYTHONPATH=DBT_LIB, PYTHONNOUSERSITE="1",
           DBT_DATABRICKS_TOKEN=dbutils.secrets.get("oil-shock", "databricks-token"),
           DBT_TARGET_PATH="/tmp/dbt_target",
           DBT_LOG_PATH="/tmp/dbt_logs",
           DBT_PROFILES_DIR=PROFILES)

def run_dbt(*args):
    r = subprocess.run([sys.executable, "-c", "from dbt.cli.main import cli; cli()",
                        *args, "--project-dir", DBT],
                       capture_output=True, text=True, env=env)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
    print("SUCCESS:", r.returncode == 0)
    return r.returncode == 0

run_dbt("debug")

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/staging/_sources.yml": """version: 2

sources:
  - name: bronze
    database: oil_shock
    schema: bronze
    description: Raw data as downloaded by the Python loaders (phase 1).
    tables:
      - name: eia_series
        description: EIA API v2 series values (prices, US balances, rigs, STEO forecasts by release month).
      - name: jodi_oil
        description: JODI-Oil World Database monthly balances by country, product and flow.
      - name: portwatch_chokepoints
        description: IMF PortWatch daily chokepoint transits. Sources UN Global Platform; IMF PortWatch.
      - name: cftc_cot
        description: CFTC disaggregated futures-only Commitments of Traders (NYMEX WTI, RBOB, ULSD).
      - name: ice_cot
        description: ICE Futures Europe Commitments of Traders (Brent, gasoil, Dubai 1st Line).
      - name: fred_observations
        description: FRED series observations (dollar, rates, CPI, industry, VIX, IMF oil prices).
      - name: fred_series_meta
        description: FRED series descriptions incl. copyright notes.
      - name: oecd_cli
        description: OECD composite leading indicators by area, monthly.
      - name: noaa_hurdat2
        description: NOAA HURDAT2 Atlantic storm tracks, 6-hourly, 1990 onward.
      - name: noaa_active_storms
        description: Snapshots of NOAA active storms, one row per storm per run.
      - name: gdelt_events
        description: Oil-relevant GDELT events, filtered at ingestion. Source The GDELT Project.
""",
"models/staging/stg_eia__series.sql": """with src as (
    select * from {{ source('bronze', 'eia_series') }}
)

select
    concat_ws('|', series_id, period, coalesce(vintage, '')) as row_key,
    series_id,
    series_label,
    route,
    frequency,
    case
        when frequency = 'monthly' then to_date(concat(period, '-01'))
        else to_date(period)
    end                                             as period_date,
    period                                          as period_raw,
    value,
    units,
    nullif(vintage, '')                             as steo_vintage,
    case when route = 'steo' then 'forecast' else 'actual' end as data_kind,
    loaded_at
from src
where value is not null
""",
"models/staging/stg_eia__series.yml": """version: 2

models:
  - name: stg_eia__series
    description: EIA series values with proper dates; actual data vs STEO forecasts (by release month).
    columns:
      - name: row_key
        description: series_id | period | vintage
        tests: [unique, not_null]
      - name: series_id
        tests: [not_null]
      - name: period_date
        tests: [not_null]
      - name: value
        tests: [not_null]
      - name: frequency
        tests:
          - accepted_values:
              arguments:
                values: ['daily', 'weekly', 'monthly']
      - name: data_kind
        tests:
          - accepted_values:
              arguments:
                values: ['actual', 'forecast']
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "stg_eia__series")

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/staging/stg_jodi__oil.sql": """select
    concat_ws('|', category, ref_area, time_period, energy_product, flow_breakdown, unit_measure) as row_key,
    category,
    ref_area                                as country_code,
    to_date(concat(time_period, '-01'))     as period_date,
    energy_product,
    flow_breakdown,
    unit_measure,
    obs_value,
    obs_value is not null                   as is_reported,
    assessment_code,
    source_file,
    loaded_at
from {{ source('bronze', 'jodi_oil') }}
""",
"models/staging/stg_portwatch__chokepoints.sql": """select
    concat_ws('|', portid, date)                                         as row_key,
    to_date(date)                                                        as transit_date,
    portid                                                               as chokepoint_id,
    portname                                                             as chokepoint_name,
    cast(get_json_object(attributes_json, '$.n_total') as int)           as ships_total,
    cast(get_json_object(attributes_json, '$.n_tanker') as int)          as ships_tanker,
    cast(get_json_object(attributes_json, '$.n_container') as int)       as ships_container,
    cast(get_json_object(attributes_json, '$.n_dry_bulk') as int)        as ships_dry_bulk,
    cast(get_json_object(attributes_json, '$.n_general_cargo') as int)   as ships_general_cargo,
    cast(get_json_object(attributes_json, '$.n_roro') as int)            as ships_roro,
    cast(get_json_object(attributes_json, '$.capacity') as double)       as capacity_total_mt,
    cast(get_json_object(attributes_json, '$.capacity_tanker') as double) as capacity_tanker_mt,
    loaded_at
from {{ source('bronze', 'portwatch_chokepoints') }}
""",
"models/staging/stg_fred__observations.sql": """select
    concat_ws('|', o.series_id, o.date)    as row_key,
    o.series_id,
    m.title                                as series_title,
    m.units,
    m.frequency,
    to_date(o.date)                        as obs_date,
    o.value,
    m.has_copyright_note,
    o.loaded_at
from {{ source('bronze', 'fred_observations') }} o
left join {{ source('bronze', 'fred_series_meta') }} m
    on o.series_id = m.series_id
where o.value is not null
""",
"models/staging/stg_oecd__cli.sql": """select
    concat_ws('|', ref_area, time_period)  as row_key,
    ref_area                               as area_code,
    area_name,
    to_date(concat(time_period, '-01'))    as period_date,
    obs_value                              as cli_value,
    loaded_at
from {{ source('bronze', 'oecd_cli') }}
where obs_value is not null
""",
"models/staging/stg_noaa__storm_tracks.sql": """select
    concat_ws('|', storm_id, obs_time)     as row_key,
    storm_id,
    name                                   as storm_name,
    to_timestamp(obs_time)                 as obs_ts,
    year(to_timestamp(obs_time))           as season,
    record_id,
    status,
    latitude,
    longitude,
    max_wind_kt,
    min_pressure_mb,
    source_file,
    loaded_at
from {{ source('bronze', 'noaa_hurdat2') }}
""",
"models/staging/stg_noaa__active_storms.sql": """select
    snapshot_at,
    storm_id,
    name                                   as storm_name,
    classification,
    intensity_kt,
    pressure_mb,
    latitude,
    longitude,
    last_update
from {{ source('bronze', 'noaa_active_storms') }}
""",
"models/staging/stg_gdelt__events.sql": """select
    globaleventid                                          as event_id,
    to_date(event_date, 'yyyyMMdd')                        as event_date,
    to_date(substr(date_added, 1, 8), 'yyyyMMdd')          as news_date,
    case when length(date_added) = 14
         then to_timestamp(date_added, 'yyyyMMddHHmmss') end as news_ts,
    datediff(to_date(substr(date_added, 1, 8), 'yyyyMMdd'),
             to_date(event_date, 'yyyyMMdd'))              as days_event_before_news,
    actor1_code, actor1_name, actor1_country,
    actor2_code, actor2_name, actor2_country,
    is_root_event,
    event_code, event_base_code, event_root_code,
    quad_class, goldstein,
    num_mentions, num_sources, num_articles, avg_tone,
    geo_type, geo_name, geo_country, geo_lat, geo_long,
    source_url,
    matched_by,
    source_file,
    loaded_at
from {{ source('bronze', 'gdelt_events') }}
""",
"models/staging/_staging.yml": """version: 2

models:
  - name: stg_jodi__oil
    description: JODI monthly balances by country, product, flow and unit.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: period_date
        tests: [not_null]
  - name: stg_portwatch__chokepoints
    description: Daily ship transits per chokepoint, by vessel type, with tanker capacity.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: ships_total
        tests: [not_null]
  - name: stg_fred__observations
    description: FRED observations with series title, units and copyright flag.
    columns:
      - name: row_key
        tests: [unique, not_null]
  - name: stg_oecd__cli
    description: OECD composite leading indicators (amplitude adjusted), monthly by area.
    columns:
      - name: row_key
        tests: [unique, not_null]
  - name: stg_noaa__storm_tracks
    description: Atlantic storm positions and strength every 6 hours (HURDAT2), 1990 onward.
    columns:
      - name: row_key
        tests: [unique, not_null]
  - name: stg_noaa__active_storms
    description: Active storm snapshots, one row per storm per ingest run.
  - name: stg_gdelt__events
    description: Oil-relevant GDELT events with event date, news (publication) date and lag between them.
    columns:
      - name: event_id
        tests: [unique, not_null]
      - name: news_date
        tests: [not_null]
      - name: matched_by
        tests:
          - accepted_values:
              arguments:
                values: ['oil_word', 'conflict_place', 'diplomacy_place']
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "staging")

In [0]:
import json
for table in ["cftc_cot", "ice_cot"]:
    row = spark.sql(f"SELECT row_json FROM oil_shock.bronze.{table} ORDER BY report_date DESC LIMIT 1").collect()[0][0]
    keys = list(json.loads(row).keys())
    wanted = [k for k in keys if any(w in k.lower() for w in
              ["open_interest_all", "prod_merc", "swap", "m_money", "other_rept", "nonrept", "market", "date", "code"])]
    print(f"\n{table}: {len(keys)} fields. Relevant:")
    print(wanted[:40])

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/staging/stg_cot__positions.sql": """{% set n = "cast(regexp_replace(get_json_object(row_json, '$.{}'), ',', '') as double)" %}

with cftc as (
    select
        'CFTC'                                         as source,
        case market_code
            when '067651' then 'wti_nymex'
            when '111659' then 'rbob_nymex'
            when '022651' then 'ulsd_nymex'
        end                                            as contract_key,
        'futures_only'                                 as report_type,
        market_name,
        to_date(report_date)                           as report_date,
        {{ n.format('open_interest_all') }}            as open_interest,
        {{ n.format('prod_merc_positions_long') }}     as producer_long,
        {{ n.format('prod_merc_positions_short') }}    as producer_short,
        {{ n.format('swap_positions_long_all') }}      as swap_long,
        {{ n.format('swap__positions_short_all') }}    as swap_short,
        {{ n.format('m_money_positions_long_all') }}   as managed_money_long,
        {{ n.format('m_money_positions_short_all') }}  as managed_money_short,
        {{ n.format('other_rept_positions_long') }}    as other_reportable_long,
        {{ n.format('other_rept_positions_short') }}   as other_reportable_short,
        {{ n.format('nonrept_positions_long_all') }}   as nonreportable_long,
        {{ n.format('nonrept_positions_short_all') }}  as nonreportable_short,
        loaded_at
    from {{ source('bronze', 'cftc_cot') }}
),

ice as (
    select
        'ICE'                                          as source,
        case
            when market_name like 'ICE Brent Crude Futures and Options%' then 'brent_ice'
            when market_name like 'ICE Brent Crude Futures%'             then 'brent_ice'
            when market_name like 'ICE Gasoil Futures and Options%'      then 'gasoil_ice'
            when market_name like 'ICE Gasoil Futures%'                  then 'gasoil_ice'
            when market_name like 'ICE Dubai%'                           then 'dubai_ice'
        end                                            as contract_key,
        case when market_name like '%and Options%' then 'futures_and_options'
             else 'futures_only' end                   as report_type,
        market_name,
        to_date(report_date)                           as report_date,
        {{ n.format('Open_Interest_All') }}            as open_interest,
        {{ n.format('Prod_Merc_Positions_Long_All') }} as producer_long,
        {{ n.format('Prod_Merc_Positions_Short_All') }} as producer_short,
        {{ n.format('Swap_Positions_Long_All') }}      as swap_long,
        {{ n.format('Swap_Positions_Short_All') }}     as swap_short,
        {{ n.format('M_Money_Positions_Long_All') }}   as managed_money_long,
        {{ n.format('M_Money_Positions_Short_All') }}  as managed_money_short,
        {{ n.format('Other_Rept_Positions_Long_All') }} as other_reportable_long,
        {{ n.format('Other_Rept_Positions_Short_All') }} as other_reportable_short,
        {{ n.format('NonRept_Positions_Long_All') }}   as nonreportable_long,
        {{ n.format('NonRept_Positions_Short_All') }}  as nonreportable_short,
        loaded_at
    from {{ source('bronze', 'ice_cot') }}
),

unioned as (
    select * from cftc
    union all
    select * from ice
)

select
    concat_ws('|', contract_key, report_type, cast(report_date as string)) as row_key,
    *,
    managed_money_long - managed_money_short       as managed_money_net,
    producer_long - producer_short                 as producer_net,
    swap_long - swap_short                         as swap_net
from unioned
""",
"models/staging/stg_cot__positions.yml": """version: 2

models:
  - name: stg_cot__positions
    description: Weekly trader positions for oil futures (CFTC NYMEX WTI/RBOB/ULSD; ICE Brent, gasoil, Dubai), one row per contract, report type and week, with net positions.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: contract_key
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ['wti_nymex', 'rbob_nymex', 'ulsd_nymex', 'brent_ice', 'gasoil_ice', 'dubai_ice']
      - name: report_date
        tests: [not_null]
      - name: open_interest
        tests: [not_null]
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "stg_cot__positions")

display(spark.sql("""
    SELECT contract_key, report_type, MIN(report_date) AS first_week, MAX(report_date) AS last_week, COUNT(*) AS weeks,
           MAX_BY(managed_money_net, report_date) AS latest_money_manager_net
    FROM oil_shock.silver.stg_cot__positions
    GROUP BY contract_key, report_type ORDER BY contract_key, report_type
"""))